# Global daily budget climatology

Day-of-year climatologies of the ACCESS-OM2 online mixed-layer temperature budget terms,
computed with `mhw3d.best_practice.compute_climatology`.

**Run this before NB01–NB03.** They open the file it writes.

## Why

The budget climatology these notebooks used up to now was a **monthly** mean
(`mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc`), linearly interpolated
to daily inside each notebook. In seasonally ice-covered regions the seasonal cycle is
*truncated*: near-flat through the ice-covered months, then a short, sharp summer excursion.
Twelve monthly values linearly interpolated cannot represent the shoulders of that cycle, so
the interpolated climatology aliased them into the daily budget anomalies — in exactly the
regions this chapter is about.

Computing the climatology directly from the daily budget output removes that aliasing, and
uses the same day-of-year construction as `om2_025_MLT_clim.nc`, so budget anomalies and MHW
detection rest on the same baseline.

## Why global

The Arctic study needs the same climatology, and a day-of-year climatology is generally
better posed than a monthly one wherever a sharp or short seasonal feature matters. One
global file serves both hemispheres; regional notebooks slice it by latitude on load.

## Why the rechunk

A day-of-year climatology is a reduction *along time*, but the daily budget files are chunked
*in time* — so each chunk contributes to essentially all 366 day-of-year slots, and its partial
aggregate is the same size as the chunk itself. There is no reduction in data volume until the
very end, and dask has to hold ~30 full-size partials to combine them. That is what exhausts
worker memory, and no amount of chunk tuning fixes it: it is the data layout, not the chunk size.

Rechunking to a **time-contiguous** layout first — the whole 30-year series for a small spatial
tile in one chunk — makes the reduction entirely chunk-local: 355 MB in, 12 MB out, per chunk,
nothing to combine. This is what `MLT_MHW_thresholds.ipynb` does to build `om2_025_MLT_clim.nc`,
and it is why that notebook runs where a direct computation does not.

`rechunker` performs that transposition out-of-core through an intermediate store with a hard
memory cap. Doing the same thing with `ds.chunk({'time': -1, ...}).to_zarr()` rebuilds exactly
the impossible single-graph rechunk we are trying to avoid — **use `rechunker`**.

Rechunking all seven raw terms at once would need far too much scratch, so section 4 loops:
rechunk one variable → compute its climatology → save it → delete the store → next. Peak scratch
is one variable's store plus its intermediate. A variable whose climatology file already exists
is skipped, so a job that dies resumes where it stopped.

`surf_to_ML` is **not** computed this way. Every step of `compute_climatology` is linear, so
`clim(surface_flux + sw_pen) == clim(surface_flux) + clim(sw_pen)` exactly, and it is derived in
section 5 for free. Section 6 checks the one thing that could break that — the two NaN masks
differing.

## Baseline period

Outputs 336–365 = 1989–2018, matching the climatology period of Holmes & Malan (2026) and of
`MLT_MHW_thresholds.ipynb`, which selects `slice("1989-01-01", "2018-12-31")` over the same
outputs. (output305 = 1958, so output *NNN* = 1958 + *NNN* − 305.)

In [ ]:
import os
import shutil
import time

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from rechunker import rechunk

from mhw3d import best_practice

%matplotlib inline

In [ ]:
import dask
from dask.distributed import Client

# Root-task overproduction is the usual way a graph like this blows its memory
# budget: the scheduler loads input chunks faster than the reductions downstream
# can consume them. Queuing the root tasks is the documented fix.
dask.config.set({'distributed.scheduler.worker-saturation': 1.0})

# Match N_WORKERS to the CPUs you actually requested, leaving one for the
# scheduler and client. On Gadi psutil reports the whole node rather than your
# PBS/ARE allocation, so set memory_limit explicitly:
#   memory_limit ≈ (your allocation × 0.9) / N_WORKERS
N_WORKERS    = 6
MEMORY_LIMIT = '38GB'

# processes=True + threads_per_worker=1 avoids netCDF4/HDF5 thread-safety races
# when workers read chunks from the same open_mfdataset files concurrently. Do
# NOT fall back to dask's default threaded scheduler: concurrent reads raise
# "NetCDF: HDF error" or corrupt the heap outright.
client = Client(processes=True, threads_per_worker=1,
                n_workers=N_WORKERS, memory_limit=MEMORY_LIMIT)
client

## 1. Configuration

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
BUDGET_SUBDIR = 'post_processed_diags/mlt_budget_online_stavg/'
MONTHLY_CLIM_FILE = (base + BUDGET_SUBDIR +
                     'mlt_budget_stavg_daily_online_output336-365_monthly_mean.ncea.nc')

OUTPUT_DIR = '/scratch/m35/nm5072/Polar_MHWs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUT_FILE = OUTPUT_DIR + 'mlt_budget_clim_daily_336-365_global.nc'

# Per-variable climatologies (kept — they are the resume points) and the
# scratch area for the rechunked stores (deleted as we go).
CLIM_DIR = OUTPUT_DIR + 'clim_parts/'
ZARR_DIR = OUTPUT_DIR + 'rechunk_tmp/'
os.makedirs(CLIM_DIR, exist_ok=True)
os.makedirs(ZARR_DIR, exist_ok=True)

# Climatology baseline: outputs 336-365 = 1989-2018
START_OUTPUT = 336
END_OUTPUT   = 365
outputs      = list(range(START_OUTPUT, END_OUTPUT + 1))

def output_to_year(o):
    return 1958 + o - 305

# ── Domain ────────────────────────────────────────────────────────────────────
# None = full global grid. Set both to restrict (e.g. -80.0 / -60.0 for the
# Antarctic strip alone).
LAT_MIN = None
LAT_MAX = None

# ── Budget terms ──────────────────────────────────────────────────────────────
# One rechunk-and-reduce pass each. 'surf_to_ML' is NOT here — it is derived in
# section 5 as clim(surface_flux) + clim(sw_pen), which is exact because every
# step of compute_climatology is linear.
RAW_TERMS = ['mlt_tendency', 'advection', 'vert_mixing',
             'entrainment', 'surface_flux', 'sw_pen', 'residual']

# What ends up in the output file.
CLIM_TERMS = RAW_TERMS + ['surf_to_ML']

# ── mhw3d climatology parameters ──────────────────────────────────────────────
# WINDOW_HALF_WIDTH = 1 means no +/-day pooling before the day-of-year mean.
# That is what mhw3d does in BOTH modes: best_practice.compute_climatology
# defaults to 1, and legacy.compute_climatology hardcodes _pool_window(da, 0).
# Only compute_threshold pools (+/-5 days), where it is needed to get enough
# samples for a 90th percentile — a 30-year per-DOY mean already has 30.
# Pooling here would also widen the effective smoothing to ~41 days once the
# 31-day smooth is applied, blunting exactly the sharp shoulders of the
# truncated polar cycle this climatology exists to resolve.
WINDOW_HALF_WIDTH = 1
SMOOTH_MEAN       = True
SMOOTH_MEAN_WIDTH = 31

# ── Rechunking ────────────────────────────────────────────────────────────────
# Target: whole time series per chunk, small spatial tiles — the layout that
# makes the day-of-year reduction blockwise. 90x90 follows MLT_MHW_thresholds.ipynb.
TARGET_YX  = 90
MAX_MEM    = '16GB'     # rechunker's hard cap on working memory
OPEN_CHUNKS = {'time': 365, 'yt_ocean': -1, 'xt_ocean': -1}   # source layout

SEC_PER_DAY = 86400.0

print(f'Baseline  : outputs {START_OUTPUT}-{END_OUTPUT} '
      f'({output_to_year(START_OUTPUT)}-{output_to_year(END_OUTPUT)})')
print(f'Output    : {OUT_FILE}')
print(f'Clim parts: {CLIM_DIR}')
print(f'Rechunk   : {ZARR_DIR}  (deleted per variable)')
print(f'Terms     : {len(RAW_TERMS)} rechunked + surf_to_ML derived')

## 2. Running as a batch job on Gadi

Section 4 is the expensive part. Submit headless so it survives disconnection:

```bash
cat > run_budget_clim.pbs << 'EOF'
#!/bin/bash
#PBS -N mlt_budget_clim
#PBS -q normal
#PBS -l walltime=12:00:00      # a GUESS — see below
#PBS -l mem=190GB
#PBS -l ncpus=7
#PBS -l storage=gdata/av17+gdata/xp65+gdata/m35+scratch/m35
#PBS -l wd
#PBS -j oe

set -euo pipefail
PROJECT=m35

module use /g/data/xp65/public/modules
module load conda/analysis3
source /g/data/${PROJECT}/${USER}/venvs/mhw3d/bin/activate

python3 -c "import mhw3d, rechunker; print('mhw3d + rechunker OK')"

jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=43200 \
    notebooks/00_budget_clim_daily.ipynb
EOF
qsub run_budget_clim.pbs
```

**Neither the walltime nor the memory has been measured against the real files.** The loop
prints the rechunk and climatology time for each term as it finishes, so the first term tells
you the real per-variable cost — check the log early and resubmit with a sensible number rather
than guessing twice. Restricting `LAT_MIN`/`LAT_MAX` in section 1 is the way to run something
smaller.

Match `N_WORKERS` to the CPUs you request. The loop is restartable: rerun the same job and
variables whose climatology file already exists are skipped.

**Scratch:** peak use is one variable's rechunked store plus rechunker's intermediate — the
store is deleted before the next variable starts. Check `lquota` before a full run.

## 3. Resolve the input files

In [ ]:
def budget_path(output):
    return f'{base}{BUDGET_SUBDIR}mlt_budget_stavg_daily_online_output{output:03d}.nc'


# Resolve which outputs actually exist — a short baseline must not pass silently.
budget_files, outputs_found, missing = [], [], []
for o in outputs:
    p = budget_path(o)
    (budget_files.append(p), outputs_found.append(o)) if os.path.exists(p) else missing.append(o)

if missing:
    print(f'WARNING: {len(missing)} of {len(outputs)} budget files missing — '
          f'outputs {missing} (years {[output_to_year(o) for o in missing]}).')
    print('The climatology will use the available years only; this is recorded in '
          'baseline_years / outputs_used in the output attributes.')
if not budget_files:
    raise FileNotFoundError(f'No daily budget files found under {base}{BUDGET_SUBDIR}')

years_found = [output_to_year(o) for o in outputs_found]
print(f'Budget files: {len(budget_files)} '
      f'(outputs {outputs_found[0]}-{outputs_found[-1]}, '
      f'years {years_found[0]}-{years_found[-1]})')

# How the source files are chunked on disk — worth knowing, since OPEN_CHUNKS
# ideally lands on a multiple of it.
probe = xr.open_dataset(budget_files[0], decode_timedelta=False)
print(f'\nStored chunking of {RAW_TERMS[0]}: '
      f'{probe[RAW_TERMS[0]].encoding.get("chunksizes")}  '
      f'dtype={probe[RAW_TERMS[0]].dtype}')
print(f'Grid: {dict(probe.sizes)}')
probe.close()

## 4. Per-variable loop — rechunk, reduce, save, delete

For each raw term:

1. open just that variable across the baseline years (lazy),
2. `rechunker` it to a time-contiguous Zarr store,
3. compute the day-of-year climatology from that store — now a blockwise reduction,
4. write the climatology to its own NetCDF in `CLIM_DIR`,
5. delete the Zarr store before moving on.

Terms whose climatology already exists are skipped, so rerunning resumes. Nothing is cast to
another dtype at any point — the source dtype carries through.

In [ ]:
def clim_part_path(term):
    return f'{CLIM_DIR}clim_{term}.nc'


def open_term(term):
    """Lazy view of one budget variable over the baseline years, subset to the domain."""
    ds1 = xr.open_mfdataset(
        budget_files,
        decode_times=True, chunks=OPEN_CHUNKS,
        combine='nested', concat_dim='time',
        coords='minimal', compat='override',      # don't re-read coords per file
        data_vars=[term], parallel=True, decode_timedelta=False,
    )[[term]]
    if LAT_MIN is not None or LAT_MAX is not None:
        ds1 = ds1.sel(yt_ocean=slice(LAT_MIN, LAT_MAX))
    return ds1


def rechunk_term(term, ds1):
    """Rechunk one variable to a time-contiguous Zarr store. Returns its path."""
    final  = f'{ZARR_DIR}{term}.zarr'
    interm = f'{ZARR_DIR}{term}_interm.zarr'
    for p in (final, interm):
        if os.path.exists(p):
            shutil.rmtree(p)

    # Explicit time length rather than -1: rechunker's target_chunks takes sizes,
    # and being explicit avoids any ambiguity about what -1 resolves to.
    target_chunks = {'time': ds1.sizes['time'],
                     'yt_ocean': min(TARGET_YX, ds1.sizes['yt_ocean']),
                     'xt_ocean': min(TARGET_YX, ds1.sizes['xt_ocean'])}
    plan = rechunk(ds1, target_chunks, MAX_MEM, final, temp_store=interm)
    plan.execute()

    shutil.rmtree(interm)          # intermediate is dead once the plan completes
    return final


t_all = time.time()
for k, term in enumerate(RAW_TERMS, start=1):
    part = clim_part_path(term)
    if os.path.exists(part):
        print(f'[{k}/{len(RAW_TERMS)}] {term}: climatology exists, skipping')
        continue

    t0 = time.time()
    print(f'[{k}/{len(RAW_TERMS)}] {term}', flush=True)

    ds1 = open_term(term)
    if k == 1:
        doy = ds1.time.dt.dayofyear
        print(f'    domain    : {dict(ds1.sizes)}')
        print(f'    yt_ocean  : {float(ds1.yt_ocean.min()):.2f} to '
              f'{float(ds1.yt_ocean.max()):.2f} (nominal deg)')
        print(f'    time span : {str(ds1.time.values[0])[:10]} to '
              f'{str(ds1.time.values[-1])[:10]}  ({ds1.sizes["time"]} days)')
        print(f'    dayofyear : {int(doy.min())}-{int(doy.max())} '
              f'(366 ⇒ the calendar includes leap days)')

    print('    rechunking ...', flush=True)
    store = rechunk_term(term, ds1)
    ds1.close()
    t_rechunk = time.time() - t0

    dsz = xr.open_zarr(store)
    clim = best_practice.compute_climatology(
        dsz[term],
        smoothMean=SMOOTH_MEAN,
        smoothMeanWidth=SMOOTH_MEAN_WIDTH,
        windowHalfWidth=WINDOW_HALF_WIDTH,
        baseline_period=None,          # files already restricted to the baseline
    )
    if k == 1:
        print(f'    reduction : {len(clim.data.__dask_graph__()):,} tasks, '
              f'output chunks {[len(c) for c in clim.chunks]}')

    t1 = time.time()
    print('    computing climatology ...', flush=True)
    clim.to_dataset(name=term).to_netcdf(part + '.tmp')
    os.replace(part + '.tmp', part)
    dsz.close()

    shutil.rmtree(store)           # free the scratch before the next variable
    print(f'    done: rechunk {t_rechunk / 60:.1f} min, '
          f'climatology {(time.time() - t1) / 60:.1f} min  → {part}', flush=True)

print(f'\nAll terms done in {(time.time() - t_all) / 60:.1f} min')

## 5. Assemble the output file

`surf_to_ML` is derived here rather than computed. The day-of-year mean and the 31-day smooth
are both linear, so summing the two climatologies is identical to computing the climatology of
the sum — and saves a whole rechunk-and-reduce pass. Section 6 checks the assumption that could
break it.

In [ ]:
clim = xr.Dataset({t: xr.open_dataset(clim_part_path(t))[t] for t in RAW_TERMS})
clim['surf_to_ML'] = clim['surface_flux'] + clim['sw_pen']

for term in CLIM_TERMS:
    clim[term].attrs.update({
        'long_name': f'Day-of-year climatology of {term}',
        'units': 'degC s-1',
    })
clim['surf_to_ML'].attrs['note'] = 'derived: clim(surface_flux) + clim(sw_pen)'

clim.attrs.update({
    'description': ('Global daily (day-of-year) climatology of ACCESS-OM2 online '
                    'mixed-layer temperature budget terms'),
    'method': ('mhw3d.best_practice.compute_climatology '
               '(github.com/ocean-mhw/mhw3d-detection), computed from a '
               'time-contiguous rechunked store'),
    'windowHalfWidth': WINDOW_HALF_WIDTH,
    'smoothMean': str(SMOOTH_MEAN),
    'smoothMeanWidth': SMOOTH_MEAN_WIDTH,
    'baseline_outputs': f'{outputs_found[0]}-{outputs_found[-1]}',
    'baseline_years': f'{years_found[0]}-{years_found[-1]}',
    'n_years_used': len(outputs_found),
    'outputs_used': ','.join(str(o) for o in outputs_found),
    'terms': ','.join(CLIM_TERMS),
    'model': 'ACCESS-OM2 0.25 deg JRA55 IAF cycle 6, online MLT budget',
    'source_files': BUDGET_SUBDIR + 'mlt_budget_stavg_daily_online_output{NNN}.nc',
    'grid_note': ('Tripolar north of ~65N: yt_ocean is nominal there. Use '
                  'geolat_t/geolon_t for Arctic region selection.'),
    'note': ('Replaces the monthly-mean budget climatology '
             '(output336-365_monthly_mean.ncea.nc), whose monthly-to-daily '
             'interpolation aliased the truncated seasonal cycle in seasonally '
             'ice-covered regions.'),
    'created_by': 'notebooks/00_budget_clim_daily.ipynb',
})

t0 = time.time()
clim.to_netcdf(OUT_FILE + '.tmp')      # write-then-rename: a killed job leaves no
os.replace(OUT_FILE + '.tmp', OUT_FILE)  # half-written file that looks complete
print(f'Saved → {OUT_FILE}')
print(f'  {os.path.getsize(OUT_FILE) / 1e9:.2f} GB in {(time.time() - t0) / 60:.1f} min')

## 6. Verify

Three checks: the file is structurally sound; the `surf_to_ML` derivation is valid; and the new
climatology differs from the one it replaces in the way the truncated-seasonal-cycle argument
predicts.

The raw day-of-year mean in the last panel is a *shape* reference, not a target — it is
unsmoothed and, if you narrowed the baseline, may cover different years. What matters is whether
the shoulders and the timing of the summer excursion in the new curve track it more closely than
the old curve does.

In [ ]:
chk = xr.open_dataset(OUT_FILE)
print(chk)

assert set(CLIM_TERMS) <= set(chk.data_vars), 'missing terms in output'
assert bool((np.diff(chk.yt_ocean.values) > 0).all()), 'yt_ocean not monotonic'
print(f'\ndayofyear : {int(chk.dayofyear.min())}-{int(chk.dayofyear.max())}')
print(f'baseline  : {chk.attrs["baseline_years"]} '
      f'(outputs {chk.attrs["baseline_outputs"]}, {chk.attrs["n_years_used"]} years)')
print(f'smoothing : windowHalfWidth={chk.attrs["windowHalfWidth"]}, '
      f'smoothMeanWidth={chk.attrs["smoothMeanWidth"]}')
for v in CLIM_TERMS:
    print(f'  {v:14s} dtype={str(chk[v].dtype):8s} '
          f'finite {float(np.isfinite(chk[v].values).mean()):.3f}   '
          f'range {float(np.nanmin(chk[v])) * SEC_PER_DAY:+.4f} to '
          f'{float(np.nanmax(chk[v])) * SEC_PER_DAY:+.4f} °C day⁻¹')

# --- is the surf_to_ML derivation valid? -------------------------------------
# clim(a + b) == clim(a) + clim(b) holds because every step is linear. The one
# thing that would break it is the two fields having different NaN masks, since
# NaN + value = NaN but the climatology of the sum would have used the value.
m_sf = np.isfinite(chk['surface_flux'].values)
m_sw = np.isfinite(chk['sw_pen'].values)
n_diff = int((m_sf != m_sw).sum())
print(f'\nsurf_to_ML derivation — NaN masks identical: {n_diff == 0}'
      + ('' if n_diff == 0 else f'  ({n_diff:,} cells differ — derivation NOT valid there)'))

In [ ]:
# ── Old (monthly-interpolated) vs new (daily) at one ice-affected point ───────
VALIDATION_TERM = 'surf_to_ML'
VAL_LAT, VAL_LON = -65.0, -71.0        # Drake Passage / Peninsula margin

sel = dict(yt_ocean=VAL_LAT, xt_ocean=VAL_LON, method='nearest')
pt = chk[VALIDATION_TERM].sel(**sel)
print(f'Validation point: yt_ocean={float(pt.yt_ocean):.2f}, '
      f'xt_ocean={float(pt.xt_ocean):.2f}')

# --- old method: 12 monthly means → daily interpolation → 31-day smooth ---
mclim = xr.open_dataset(MONTHLY_CLIM_FILE)
mclim['surf_to_ML'] = mclim['surface_flux'] + mclim['sw_pen']
mpt = mclim[VALIDATION_TERM].sel(**sel).compute()

by_month = mpt.groupby('time.month').mean('time').assign_coords(month=np.arange(1, 13))
padded = xr.concat([by_month.sel(month=12).expand_dims(month=[0]),
                    by_month,
                    by_month.sel(month=1).expand_dims(month=[13])], dim='month')
old_curve = (padded.interp(month=np.linspace(1, 12, 365))
             .rolling(month=31, center=True, min_periods=1).mean().values) * SEC_PER_DAY

new_curve = pt.sel(dayofyear=slice(1, 365)).values * SEC_PER_DAY

# --- raw day-of-year mean straight off the source files, unsmoothed ---
raw_ds = open_term('surface_flux')
raw_sw = open_term('sw_pen')
raw = ((raw_ds['surface_flux'].sel(**sel) + raw_sw['sw_pen'].sel(**sel))
       .groupby('time.dayofyear').mean().compute()) * SEC_PER_DAY

doy_axis = np.arange(1, 366)
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(raw.dayofyear, raw.values, color='0.6', lw=1,
             label=f'raw DOY mean, {years_found[0]}–{years_found[-1]} (unsmoothed)')
axes[0].plot(doy_axis, old_curve, color='tab:orange', lw=2,
             label='old: monthly → daily interp + 31-day smooth')
axes[0].plot(doy_axis, new_curve, color='tab:blue', lw=2,
             label='new: mhw3d daily climatology')
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_ylabel('°C day⁻¹')
axes[0].legend(fontsize=9)
axes[0].set_title(f'{VALIDATION_TERM} climatology — '
                  f'{float(pt.yt_ocean):.2f}°, {float(pt.xt_ocean):.2f}°', fontsize=12)

axes[1].plot(doy_axis, new_curve - old_curve, color='tab:red', lw=1.5)
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_ylabel('new − old\n(°C day⁻¹)')
axes[1].set_xlabel('Day of year')
axes[1].set_xlim(1, 365)
plt.tight_layout()

rms = float(np.sqrt(np.nanmean((new_curve - old_curve) ** 2)))
rng = float(np.nanmax(new_curve) - np.nanmin(new_curve))
print(f'RMS(new − old)                       : {rms:.4f} °C day⁻¹')
print(f'Seasonal range of new                : {rng:.4f} °C day⁻¹')
print(f'RMS difference as % of seasonal range: {100 * rms / rng:.1f}%')

**Grid note.** ACCESS-OM2 is tripolar north of ~65°N, where `yt_ocean` is a nominal coordinate
rather than true latitude. This does not affect the climatology — it is a pointwise day-of-year
reduction — but Arctic *region selection* in the analysis notebooks should use the 2D
`geolat_t`/`geolon_t` fields, not `yt_ocean`.

**Cleanup.** `CLIM_DIR` still holds the per-variable climatologies. They are the resume points,
so keep them until you are happy with `OUT_FILE`, then remove that directory and `ZARR_DIR`.

In [ ]:
client.close()